In [ ]:
import ee
import geemap

import types

import math
from datetime import datetime
import time
from pandas import Timestamp
import pandas as pd

import geopandas as gpd
import json

import shapely


"""
Code to ensure that the content of a remote GEE asset is the same as the content saved locally
"""

GDF_NAME = "pastis_metadata"
PATH_TO_DATA = f"../gdfs/{GDF_NAME}.gpkg"

ID_KEY = "id"
ID_DATATYPE = "int64"

N = 100


In [62]:
PROJECT_ID = "hedgementation"

try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

In [63]:
table = ee.FeatureCollection(f"projects/{PROJECT_ID}/assets/{GDF_NAME}")
table.limit(10)

In [64]:
try:
    gdf = gpd.read_file(PATH_TO_DATA, engine='pyogrio', use_arrow=True)
except:
    gdf = gpd.read_file(PATH_TO_DATA)

In [ ]:
sample = gdf.sample(n=N).to_crs("EPSG:4326").copy()

In [ ]:
def find_matching(id, geodataframe):
    result = geodataframe[geodataframe[ID_KEY] == id] 
    if len(result) > 0:
        if len(result) > 1:
            print(f"Found {len(result)} GDF rows with cleabs {id}")
        return result.iloc[0]
    return -1

def create_geometry_check(current_feature, current_gdf_row: gpd.GeoSeries):
    geom = current_gdf_row["geometry"]
    geom1 = ee.Geometry(current_feature["geometry"])
    geom2 = ee.Geometry(json.loads(shapely.to_geojson(geom)))

    buffer1 = geom1.buffer(0.1)
    buffer2 = geom2.buffer(0.1)
    intersection = buffer1.intersection(buffer2)
    union = buffer1.union(buffer2)
    geometry_check = intersection.area().divide(union.area()).gt(0.95)

    return geometry_check


def check_row_equality(current_feature, current_gdf_row, columns_gdf, column_types, geometry_check):
    retval = True
    id = current_feature["properties"][ID_KEY]
     
    for k in columns_gdf:
        if pd.api.types.is_datetime64_any_dtype(column_types[k]):
            feature_val = Timestamp(current_feature["properties"][k])
            column_val = Timestamp(current_gdf_row[k])
        else:
            feature_val = current_feature["properties"][k]
            column_val = current_gdf_row[k]

        if feature_val != column_val and not(pd.isnull(feature_val) and pd.isnull(column_val)):
            print(f"For {id}, the value of {k} is different: {feature_val} vs {column_val}")
            retval = False
    
    if not geometry_check:
        eegeom = current_feature["geometry"]
        gdfgeom = json.loads(shapely.to_geojson(current_gdf_row["geometry"]))
        print(f"For {id}, the geometry check was failed.")
        print(f"EE Feature geometry: {eegeom}")
        print(f"GeoPandas Row Geometry: {gdfgeom}")
        retval = False
    return retval

def check_table_equality(feature_collection: ee.FeatureCollection, geodataframe: gpd.GeoDataFrame):
    geodataframe[ID_KEY] = geodataframe[ID_KEY].astype(ID_DATATYPE)
    ids = list(geodataframe[ID_KEY])
    feature_collection = feature_collection.filter(ee.Filter.inList(ID_KEY,ids)).getInfo()
    retval = True

    features = feature_collection["features"]

    fc_size = len(features)
    if len(geodataframe) != fc_size:
        retval = False
        if len(geodataframe) > fc_size:
            print("Geodataframe is larger than filtered feature collection. There are rows missing.")
        else:
            print("Geodataframe is smaller than filtered feature collection. There are duplicate rows.")
        print(f"GDF length {len(geodataframe)} vs FC length {len(feature_collection)}")

    columns_fc = [c for c in list(feature_collection["columns"].keys()) if c != "system:index"]
    columns_gdf = [c for c in list(geodataframe.columns) if c != "geometry"]

    def check_keys(list1, list2, name1, name2):
        if not all(c in set(list1) for c in list2):
            print(f"The following columns are present in {name2}, but not {name1}:{[c for c in columns_fc if c not in columns_gdf]}")
            return False
        return True
    
    retval = retval and check_keys(columns_fc, columns_gdf, "FC", "GDF")
    retval = retval and check_keys(columns_gdf, columns_fc, "GDF", "FC")

    
    feature_row_pairs = [(f, find_matching(f["properties"][ID_KEY], geodataframe)) for f in features]
    geometry_checks = {f["properties"][ID_KEY]: create_geometry_check(f,r) for f,r in feature_row_pairs}
    geometry_checks = ee.Dictionary(geometry_checks).getInfo()

    passed_rows = 0
    failed_rows = 0
    for f,r in feature_row_pairs:
        result = check_row_equality(f,r,columns_gdf,geodataframe.dtypes,geometry_checks[str(f["properties"][ID_KEY])])
        if result:
            passed_rows += 1
        else:
            failed_rows += 1
        retval = retval and result

    print(f"Of a total of {passed_rows + failed_rows} rows, {passed_rows} passed the checks while {failed_rows} failed.")
    if retval:
        print("All rows passed all checks.")

    return retval

check_table_equality(table, sample)

{'10025': 1, '10058': 1, '10092': 1, '10106': 1, '10110': 1, '10118': 1, '10139': 1, '10141': 1, '10168': 1, '10208': 1, '10209': 1, '10216': 1, '10260': 1, '10283': 1, '10305': 1, '10351': 1, '10362': 1, '10401': 1, '10409': 1, '10430': 1, '10445': 1, '10447': 1, '10455': 1, '10458': 1, '10494': 1, '10538': 1, '20078': 1, '20082': 1, '20090': 1, '20154': 1, '20171': 1, '20177': 1, '20183': 1, '20189': 1, '20246': 1, '20268': 1, '20270': 1, '20285': 1, '20291': 1, '20362': 1, '20447': 1, '20546': 1, '20593': 1, '20635': 1, '30003': 1, '30017': 1, '30048': 1, '30053': 1, '30067': 1, '30132': 1, '30141': 1, '30159': 1, '30193': 1, '30197': 1, '30258': 1, '30262': 1, '30288': 1, '30317': 1, '30329': 1, '30360': 1, '30365': 1, '30387': 1, '30427': 1, '30511': 1, '30522': 1, '30552': 1, '30560': 1, '30579': 1, '30635': 1, '30638': 1, '30674': 1, '30680': 1, '30693': 1, '30697': 1, '30699': 1, '30709': 1, '30720': 1, '30722': 1, '40001': 1, '40002': 1, '40012': 1, '40024': 1, '40058': 1, '40

True